# Lab 06 · Multi-agent debate: propose, challenge, agree

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · 15–20 minute lab

## Multi-agent debate: several agents, one decision

**So far:** earlier labs used the lecture material to propose service boundaries and resilience choices. Now ShopEasy support needs a screen that searches orders, payments, and deliveries by customer name (Microservices slide 30A).

**New:** *multi-agent debate*: several agents answer the same question, challenge each other's claims, and a fixed procedure turns their answers into one decision. Our example is an *architecture council*. Three architects with different stances, each running on a different **model family**, answer **alone and in parallel**. Then a *chair* runs the debate by fixed rules:

1. It challenges every claim that cites no source, and gives the author one chance to revise.
2. It counts votes with valid citations; a two-thirds majority decides.
3. It records the dissent, because a minority view can be right.
4. With no majority, it escalates to the human instead of guessing.

Every message is a small dictionary with a `round`, a `speaker`, and a `kind`. The list of all messages, the *transcript*, is the council's shared state: it shows who said what, based on which source.

The chair here is plain code, so its rules are easy to read and test. A chair can also be an agent, but keep its rules (who speaks, how many rounds, what counts as a majority) in code.

**At a glance**

![Lab 06 at a glance: Shared sources (M27 · M28 · case), then Minimalist (Qwen) or Safeguard (DeepSeek) or Modernizer (Gemini), then valid citations?, then Challenge + revise (one round), then Chair counts (⅔ of registered peers), then Decision (+ dissent kept) or Escalate (no majority).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/06.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 27, 28, 30.

**Recall:** API composition versus CQRS; Eventual consistency and read-model cost.

Before running the example:

1. Why is a customer-name search different from a lookup by order ID?
2. Which cost of CQRS must the council record?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

Parallel proposals preserve independent views. A coded chair limits discussion and aggregates proposals; agreement is not proof of correctness.

CQRS supports search but adds a read model and consistency lag; composition has different costs.

| Key term | Meaning |
| --- | --- |
| **Multi-agent system** | Several model-driven agents with separate roles that work on one task. |
| **Independence** | Each agent answers without seeing the others first, so its view is its own. |
| **Grounding** | Supporting a claim with a source that everyone can check. |
| **Escalation** | Handing the decision to a person when the agents cannot agree. |

## Pattern · Debate-Based Cooperation

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Voting-Based Cooperation, Parallel Fan-Out, Shared Memory.*

| | |
| --- | --- |
| **Context** | Hard questions hold competing values and assumptions that become clearer through structured disagreement. |
| **Problem** | Independent answers never confront one another, so weak claims and hidden trade-offs survive aggregation. |
| **Solution** | Let agents with contrasting positions respond to one another over bounded rounds, using a shared transcript, then ask a judge for a structured verdict. |
| **Use when** | Trade-offs are genuinely contested; reasoning quality matters more than speed; a transcript is useful evidence. |
| **Trade-off** | Debate surfaces assumptions, but it costs many tokens, and persuasive rhetoric can outweigh truth. |
| **Flow** | Positions → Debate K rounds → Judge → Verdict |
| **Quality attributes** | Benefits: Reliability, Accuracy, Auditability. Liabilities: Latency, Cost, Accuracy, Scalability, Predictability. |
| **In this lab** | The three proposals are a *Parallel Fan-Out*, and the transcript is the debate's *Shared Memory*. The chair is a judge with fixed rules: it challenges claims without a source for one round, then aggregates votes with valid citations, as in *Voting-Based Cooperation*. Requiring sources guards against the pattern's known risk: persuasion without evidence. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Shared sources and a citation rule

All architects see the same sources: two lecture notes (M27 and M28) and two ShopEasy facts. Equal evidence keeps the debate fair. In your project, retrieve the sources with Lab 02's vector database.

`citation_valid` is a citation-validity check: a proposal may vote only if it cites at least one known source. This checks IDs, not whether a source supports a claim; students must read that claim against the slide.

In [ ]:
sources = {  # every architect sees the same evidence, so the debate is fair
    "M27": "API composition joins small results by id; broad cross-service search is a poor fit.",
    "M28": "CQRS supports rich queries with a separate read model; the view is eventually consistent.",
    "C1": "ShopEasy support must search orders, payments and deliveries by customer name.",
    "C2": "Support accepts a slightly stale view, but the team wants few moving parts.",
}
evidence = "\n".join(f"[{sid}] {text}" for sid, text in sources.items())

def citation_valid(message):
    """Check citation IDs only; factual support still requires review."""
    return bool(message["evidence"]) and set(message["evidence"]) <= set(sources)

print(citation_valid({"evidence": ["M28", "C1"]}), citation_valid({"evidence": []}), citation_valid({"evidence": ["M99"]}))

## Step 2 · Three architects on three model families

The architects share one job and one set of sources, but each has its own stance and its own model family. The three stances are examples: in your project, design stances that fit your scenarios. Three personas on one model are not enough: one model brings one set of blind spots, so the course project requires at least three families. Each architect runs on its own provider: Qwen on FreeInference, DeepSeek, and Gemini. `PROVIDERS` in `coursekit` names the model family behind each provider, and the assert checks that the council has at least three. `asks` holds one `ask` function per architect, each with its own call budget. `say` builds every prompt the same way: the role and stance, the sources, and then the task.

Practice mode needs no keys. For live mode, save three keys in Colab Secrets: `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY`. This lab ignores the `PROVIDER` setting in the setup cell, because every architect has its own provider. Check each provider's pricing and quotas before class.

In [ ]:
from coursekit import PROVIDERS

council = {  # same job and sources; a different stance and model provider for each architect
    "Minimalist": {"provider": "freeinference", "stance": "Prefer the simplest design that meets the requirements."},
    "Safeguard": {"provider": "deepseek", "stance": "Prefer low-risk designs; weigh operations and failures."},
    "Modernizer": {"provider": "gemini", "stance": "Prefer designs that keep search responsive as order history grows."},
}
families = {name: PROVIDERS[a["provider"]]["family"] for name, a in council.items()}
assert len(set(families.values())) >= 3, "The council needs at least three model families."
asks = {name: make_ask(mode=MODE, provider=a["provider"]) for name, a in council.items()}
print(families)

def say(name, task, example):
    """Ask one architect: its role and stance, the shared sources, then the task."""
    role = f"You are the {name} on an architecture council. {council[name]['stance']}"
    return asks[name](f"{role}\nSources:\n{evidence}\n{task}", example=example)

## Step 3 · Propose alone, in parallel

Each architect answers without seeing the others. Models tend to agree with what they have just read, so a proposal written after reading another one is not an independent opinion. `ThreadPoolExecutor` starts the three calls at the same time. (The course client limits how many requests go to one service at once. FreeInference's free tier takes one at a time, so architects that share it wait their turn; architects on different services really do run in parallel.) The chair turns each JSON reply into a message by adding the round, the speaker, and the kind.

In practice mode, the Modernizer's scripted claim cites no source. Step 4 shows what the chair does with it.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

REPLY = 'Reply with JSON only, no code fences: {"option": "...", "claim": "one sentence", "evidence": ["ids"]}'
scripted = {
    "Minimalist": '{"option": "CQRS", "claim": "Rich cross-service search fits a separate read model.", "evidence": ["M28", "C1"]}',
    "Safeguard": '{"option": "API composition", "claim": "A composer needs fewer moving parts, but search may join large results.", "evidence": ["C2"]}',
    "Modernizer": '{"option": "CQRS", "claim": "CQRS guarantees a perfectly fresh answer.", "evidence": []}',
}

def propose(name):
    """One architect answers alone: the prompt never contains another proposal."""
    task = f"Choose API composition or CQRS for ShopEasy support search. {REPLY}"
    return {"round": 1, "speaker": name, "kind": "proposal", **json.loads(say(name, task, scripted[name]))}

with ThreadPoolExecutor() as pool:  # the three architects work at the same time
    proposals = {message["speaker"]: message for message in pool.map(propose, council)}
for m in proposals.values():
    print(f"{m['speaker']:<11} {m['option']:<17} {str(m['evidence']):<19} {m['claim']}")

## Step 4 · Challenge unsupported claims

The chair checks every proposal with `citation_valid`. For a claim without a source, it picks a critic, preferring an architect who chose a different option, and asks for a one-sentence challenge. The author gets one chance to revise: it can change its option, or keep it with a claim that cites sources. Allowing one challenge per claim keeps the debate bounded, so it always ends. Each message is appended to `transcript`.

In [ ]:
transcript = list(proposals.values())  # shared state: every message, in order
critique = "M28 says the read view lags behind writes. Which source supports a freshness guarantee?"
revision = '{"option": "CQRS", "claim": "Rich cross-service search fits a separate read model.", "evidence": ["M28", "C1"]}'
for name, m in list(proposals.items()):
    if citation_valid(m):
        continue
    others = [n for n in council if n != name]
    critic = next((n for n in others if proposals[n]["option"] != m["option"]), others[0])  # prefer an opponent
    challenge = say(critic, f"In one sentence, challenge this claim by the {name}: {m['claim']}", critique)
    transcript.append({"round": 2, "speaker": critic, "kind": "challenge", "target": name, "text": challenge})
    task = f"You claimed: {m['claim']}\nChallenge: {challenge}\nRevise your proposal. {REPLY}"
    proposals[name] = {"round": 2, "speaker": name, "kind": "revision", **json.loads(say(name, task, revision))}
    transcript.append(proposals[name])
    print(f"{critic} challenges {name}: {challenge}")
    print(f"{name} revises: {proposals[name]['option']}, {proposals[name]['claim']} {proposals[name]['evidence']}")

## Step 5 · Count votes with valid citations, keep the dissent

Only proposals with valid citations vote. With a two-thirds majority, the chair decides and keeps every dissenting proposal, with its sources, in the outcome. Without one, the chair escalates to the human. Printing the transcript replays the whole debate.

In [ ]:
from collections import Counter

votes = Counter(m["option"] for m in proposals.values() if citation_valid(m))  # claims without valid citation IDs do not vote
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = 3 * count >= 2 * len(council)  # the stopping rule: a two-thirds majority
outcome = {"round": 3, "speaker": "Chair", "kind": "decision" if decided else "escalation",  # no guessing
           "option": option if decided else None, "votes": dict(votes),
           "dissent": [m for m in proposals.values() if decided and m["option"] != option]}
transcript.append(outcome)
for m in transcript:
    print(f"round {m['round']}  {m['speaker']:<11} {m['kind']:<10} {m.get('text') or m['option']}")
print("Votes:", outcome["votes"], "| dissent:", [(m["speaker"], m["claim"]) for m in outcome["dissent"]])

## What you should see

The Modernizer's claim that CQRS guarantees freshness has no citation. The Safeguard challenges it using M28: the read view is eventually consistent. The revision keeps CQRS, cites M28 and C1, and the chair decides 2–1. The API-composition dissent remains visible: a separate read model costs more moving parts. The five scripted calls demonstrate the control flow; they are not evidence that a council improves real answers.

## Try it

1. Remove the revision's citations. Only one vote supports each option, so the chair escalates.
2. Change C1 to the slide 30C scenario: low-traffic order details, lookup by id across three services. Why does M27 now favour API composition? Update practice replies too.
3. Give a false claim a real citation ID. The automated check accepts its ID; does the slide actually support it?
4. In live mode, compare each first proposal with the final choice. Record cost and dissent even when no one changes their mind.

## Takeaway

Independent proposals, bounded critique, aggregation, and preserved dissent make an agent council inspectable. The architecture lesson is separate: API composition suits small lookups; CQRS supports rich queries but adds a read model and eventual consistency.